# T01: Die Baseline-RAG-Pipeline

Dieses Notebook baut Schritt für Schritt eine funktionierende RAG-Pipeline auf den Dokumenten der PrimePack AG auf und testet dann, wo sie scheitert. Die Zellen von oben nach unten ausführen; Fragen und Parameter dürfen frei geändert werden.

| Notebook | Thema |
|---|---|
| **T01** | Baseline-Pipeline (dieses Notebook) |
| T02, T03 | Ingestion, Chunking, multimodale Dokumente |
| T04, T05 | Antwortqualität messen |
| T06 | Strukturierte Ausgaben |
| T07 bis T09 | Retrieval, Konversation, Query-Techniken |
| T10 bis T14 | Tools und Agenten |

### Warum Code, Prompts und Dokumente auf Englisch sind

Die Erklärungen sind auf Deutsch, alles, was ausgeführt oder an das Modell geschickt wird, bleibt auf Englisch:

- **Dokumente**: Die Dateien in `data/` sind englisch. Fragen und Prompts in derselben Sprache wie die Dokumente führen meist zu besseren Treffern beim Retrieval.
- **Prompts und Beispielfragen**: Sie sind auf die Dokumente und das Testset in T04 abgestimmt; so bleiben die Ergebnisse mit der englischen Version vergleichbar. Wo ein Prompt vorkommt, steht im Text eine deutsche Übersetzung. Sie dient nur dem Verständnis und wird nicht an das Modell geschickt.
- **Code**: Bezeichner, Ausgaben und Fehlermeldungen von Bibliotheken sind englisch.

Fragen auf Deutsch funktionieren trotzdem; das LLM antwortet dann meist auf Deutsch.

## Warum RAG?

Die **PrimePack AG** kauft Verpackungen (Paletten, Kartons, Klebeband) bei mehreren Lieferanten ein. Mitarbeitende müssen Fragen wie diese beantworten:

> *What is the GWP of the Logypal 1 pallet, and is the figure verified?* (Wie hoch ist das GWP der Palette Logypal 1, und ist der Wert verifiziert?)\
> *Can we tell a customer that the tesa tape is PFAS-free?* (Dürfen wir einem Kunden sagen, dass das tesa-Klebeband PFAS-frei ist?)

Ein LLM allein kann sie nicht zuverlässig beantworten:

| Problem | Warum es wichtig ist |
|---|---|
| Kein Zugriff auf interne Dokumente | Die Antwort ist nicht in den Trainingsdaten |
| Halluzination | Das Modell erfindet unter Umständen plausible Zahlen |
| Keine Nachweise | Antworten lassen sich keiner Quelle zuordnen |

**Retrieval-Augmented Generation (RAG)** durchsucht zuerst die Dokumente und lässt das LLM dann nur auf Basis der gefundenen Auszüge antworten:

```
 Documents ──► Chunker ──► Embedder ──► Vector DB
                                              │
 User query ────────-──► Embedder ─────►  Retriever ──► Top-k Chunks
                                                              │
                                                          LLM + Prompt
                                                              │
                                                       Answer + Sources
```

RAG verlagert das Problem von der Halluzination zur **Retrieval-Qualität**: Wird der richtige Chunk nicht gefunden, ist die Antwort trotzdem falsch. Darum geht es in T02 bis T09.

## Grundbegriffe

- **Chunk**: ein kurzer Auszug aus einem Dokument (ein Abschnitt eines PDFs, ein Tabellenblatt) mit `title`, `content`, `mime_type` und `metadata` wie `source_file`. Chunks sind die Einheit für Indexierung und Retrieval.
- **Embedding**: ein Vektor, der die Bedeutung eines Textes darstellt; ähnliche Texte erhalten ähnliche Vektoren. Wir verwenden `text-embedding-3-small` von OpenAI (1536 Dimensionen).
- **Vector Store**: speichert die Embeddings der Chunks und liefert die `top_k` Chunks, die einem Query-Embedding am nächsten sind. Wir verwenden ChromaDB.
- **Retriever**: sucht mit `retrieve(query)` die passenden Chunks im Vector Store.
- **RAG-Agent**: sucht Chunks zu einer Frage, übergibt sie dem LLM als `<source>`-Auszüge und gibt die Antwort mit ihren Quellen zurück.

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos).

In [ ]:
import sys
from collections import Counter
from statistics import median

import numpy as np
from loguru import logger

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from ki_kompass_rag.baseline_rag import (
    EMBEDDING_MODEL,
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    build_vector_store,
    inspect_retrieval,
    load_chunks,
)
from ki_kompass_rag.ingestion import EMBEDDING_TOKEN_LIMIT, estimate_tokens

logger.remove()
logger.add(sys.stderr, level="WARNING")

llm = build_llm("openai")

## Das LLM allein

Ein LLM weiss nur, was in seinen Trainingsdaten stand. Zuerst eine allgemeine Frage:

In [ ]:
async def ask_llm(question: str) -> str:
    message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=question)]
    )
    response = await llm.generate([message])
    return "".join(c.text or "" for c in response.content)


print(await ask_llm("What does GWP stand for, and what unit is it measured in?"))

Jetzt eine Frage zum Sortiment der PrimePack AG:

In [ ]:
print(
    await ask_llm(
        "What is the GWP of the Logypal 1 pallet sold by PrimePack AG, and is the "
        "figure third-party verified? Provide the link to PrimePack AG's website."
    )
)

Die PrimePack AG ist fiktiv. Eine konkrete Zahl oder ein Link ist halluziniert; "I don't know" ist ehrlich, aber nutzlos. GPT-4o-mini lehnt häufiger ab als kleine lokale Modelle, die eher eine Antwort erfinden. So oder so braucht der verifizierte Wert die tatsächliche EPD.

### OpenAI oder ein lokales Modell?

| | OpenAI API (`gpt-4o-mini`) | Lokal über Ollama (`mistral-nemo:12b`) |
|---|---|---|
| Daten | Gehen an OpenAI | Bleiben auf dem Rechner |
| Qualität | Befolgt Anweisungen zuverlässig | Schwächer bei komplexen Regeln |
| Kosten | Pro Token | Hardware und Strom |

Der Workshop verwendet OpenAI. `build_llm("ollama")` liefert ein lokales Modell mit derselben Schnittstelle.

### Kostenschätzung

`gpt-4o-mini` kostet $0.15 pro 1 Mio. Input-Tokens und $0.60 pro 1 Mio. Output-Tokens ([Preise](https://developers.openai.com/api/docs/pricing/)). Ein Token entspricht etwa 4 Zeichen englischen Textes. Eine RAG-Anfrage sendet den System-Prompt, `top_k` Chunks und die Frage; angenommen werden ~400 Tokens pro Chunk und eine Antwort von 150 Tokens:

In [ ]:
INPUT_PRICE = 0.15 / 1_000_000  # USD per token
OUTPUT_PRICE = 0.60 / 1_000_000

tokens_per_chunk = 400
question = "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?"
input_tokens = (
    estimate_tokens(SYSTEM_PROMPT)
    + RETRIEVER_TOP_K * tokens_per_chunk
    + estimate_tokens(question)
)
output_tokens = 150
cost = input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE

print(f"~{input_tokens:,} input + {output_tokens} output tokens: ${cost:.5f} per query")
for n in (1_000, 100_000):
    print(f"{n:>9,} queries/day: ${cost * n:,.2f}")

Die Input-Tokens überwiegen, die Kosten steigen also mit `top_k` und der Chunk-Grösse.

## Schritt 1: Dokumente laden und chunken

`load_chunks()` wandelt jede Datei in `data/` in Markdown um und teilt sie an den Überschriften auf (PDFs mit docling, Excel ein Chunk pro Tabellenblatt). Abschnitte über `MAX_CHUNK_CHARS` (4000 Zeichen) werden zusätzlich an Absätzen geteilt. Dateien über 20 MB werden übersprungen. Auf der CPU dauert das einige Minuten. T02 vergleicht Parser und Chunking-Strategien.

In [ ]:
chunks = load_chunks()

lengths = [len(c.content) for c in chunks]
files = Counter(c.metadata["source_file"] for c in chunks)
too_long = sum(estimate_tokens(c.content) > EMBEDDING_TOKEN_LIMIT for c in chunks)
print(f"{len(chunks)} chunks from {len(files)} files")
print(f"Median length: {median(lengths):.0f} chars, longest: {max(lengths):,} chars")
print(f"Over the {EMBEDDING_TOKEN_LIMIT}-token embedding limit: {too_long}")

In [ ]:
for c in chunks[:3]:
    print(f"[{c.metadata['source_file']}] {c.title!r} ({len(c.content)} chars)")
    print(f"{c.content[:200].strip()!r}\n")

## Schritt 2: Chunks embedden und den Vector Store bauen

Der `content` jedes Chunks wird embedded und in einer ChromaDB-Collection unter `backend/db/data_vs.db` gespeichert. `reset=True` baut den Store neu auf, damit er immer zu den aktuellen Chunks passt; T02 bis T07 verwenden ihn ohne erneutes Embedding.

| | `text-embedding-3-small` (hier verwendet) | Sentence-Transformers-Modelle, z. B. `all-MiniLM-L6-v2` |
|---|---|---|
| Dimensionen | 1536 | 384 bis 1024 |
| Kontextlimit | 8191 Tokens | 256 bis 8192 Tokens, je nach Modell |
| Kosten | ~$0.02 pro 1 Mio. Tokens | Gratis, läuft lokal (langsam auf der CPU) |
| Daten | Gehen an OpenAI | Bleiben lokal |

Jedes Modell aus dem [MTEB-Leaderboard](https://huggingface.co/spaces/mteb/leaderboard), das mit `sentence-transformers` funktioniert, lässt sich über `SentenceTransformerEmbeddings(model_name)` verwenden.

ChromaDB läuft eingebettet (ohne Server) und eignet sich für bis zu ~100k Chunks mit einem einzigen schreibenden Prozess. Für gleichzeitigen Zugriff und SQL-Filter hat das Toolkit zusätzlich `PGVectorStore` (PostgreSQL + pgvector).

In [ ]:
embedding_model = OpenAIEmbeddings(EMBEDDING_MODEL)
vector_store = await build_vector_store(chunks, embedding_model, reset=True)
print(f"{vector_store.collection.count()} chunks in the store")

### Ähnlichkeit im Embedding-Raum

Die Kosinus-Ähnlichkeit misst, wie ähnlich die Richtung zweier Embeddings ist: 1 heisst gleiche Richtung, um 0 kein Zusammenhang. Die Sätze ändern und vergleichen. (Der [Embedding Projector](https://projector.tensorflow.org) gibt einen visuellen Eindruck.)

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


anchor = "carbon footprint of a pallet"
others = [
    "GWP value for the Logypal 1",
    "PFAS-free tape declaration",
    "the annual report of a software firm",
]
vectors = await embedding_model.get_embeddings([anchor, *others])
for text, vector in zip(others, vectors[1:], strict=True):
    print(f"{cosine(vectors[0], vector):.3f}  {anchor!r} vs {text!r}")

### Embedding-Modelle vergleichen

Ein schneller Test: Wie viel ähnlicher ist ein relevanter Chunk der Frage als ein irrelevanter? Je grösser der Abstand, desto besser trennt das Modell nützliche von störenden Chunks. `all-MiniLM-L6-v2` wird beim ersten Aufruf heruntergeladen (~90 MB). Ein Beispiel ist keine Evaluation; T04 misst das Retrieval richtig.

In [ ]:
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)

query = "What is the carbon footprint of the Logypal 1 pallet?"
relevant = (
    "Logypal 1 internal LCA (2020, non-verified): GWP 4.1 kg CO2e per pallet "
    "over a 50-trip lifetime."
)
irrelevant = (
    "All suppliers must comply with applicable environmental regulations "
    "and report annually on progress."
)
models = {
    "all-MiniLM-L6-v2 (local, 384-dim)": SentenceTransformerEmbeddings(
        "sentence-transformers/all-MiniLM-L6-v2"
    ),
    "text-embedding-3-small (API, 1536-dim)": embedding_model,
}

print(f"{'model':<40}{'relevant':>10}{'irrelevant':>12}{'gap':>8}")
for name, model in models.items():
    q, rel, irr = await model.get_embeddings([query, relevant, irrelevant])
    print(
        f"{name:<40}{cosine(q, rel):>10.3f}{cosine(q, irr):>12.3f}"
        f"{cosine(q, rel) - cosine(q, irr):>8.3f}"
    )

## Schritt 3: Retrieval prüfen

Sind die gefundenen Chunks falsch, ist die Antwort falsch, egal wie gut das LLM ist. Deshalb zuerst das Retrieval prüfen, bevor man dem Modell die Schuld gibt.

`inspect_retrieval()` zeigt die Top-k-Chunks mit ihrem `score`, der Kosinus-Ähnlichkeit zwischen Frage und Chunk (höher heisst ähnlicher).

In [ ]:
results = await inspect_retrieval(
    "What materials is the Logypal 1 pallet made from?",
    vector_store,
    embedding_model,
    top_k=RETRIEVER_TOP_K,
)

### Ein Produkt ausserhalb des Sortiments

Kein Dokument erwähnt eine "Lara pallet". Die Scores mit der Frage oben vergleichen.

In [ ]:
results = await inspect_retrieval(
    "What materials is the Lara pallet made from?",
    vector_store,
    embedding_model,
    top_k=RETRIEVER_TOP_K,
)

Der Retriever kennt kein "kein Treffer": Er liefert immer die nächstgelegenen Chunks. Die Scores sind tiefer, aber das LLM erhält trotzdem Chunks zu anderen Paletten.

## Schritt 4: Den RAG-Agenten bauen

`build_agent()` verbindet einen `VectorStoreRetriever` (Vector Store + Embedding-Modell), das LLM und einen System-Prompt. Wir beginnen mit einem bewusst minimalen Prompt; die Tests unten zeigen, was ihm fehlt.

`BASIC_PROMPT` übersetzt:

> Du bist ein hilfreicher KI-Assistent, spezialisiert auf Nachhaltigkeit und Produkt-Compliance für die PrimePack AG.
>
> Du erhältst Dokumentauszüge, die für die Frage der Nutzerin oder des Nutzers relevant sind. Formuliere die bestmögliche Antwort und verwende dabei nur die Informationen aus diesen Auszügen.

In [ ]:
BASIC_PROMPT = (
    "You are a helpful AI assistant specialised in sustainability and product "
    "compliance for PrimePack AG.\n\n"
    "You will receive document excerpts relevant to the user's question. Produce "
    "the best possible answer using only the information in those excerpts."
)

agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=BASIC_PROMPT,
    number_query_expansion=0,  # see T09
)

## Schritt 5: Eine Frage stellen

`ask()` sucht die Top-k-Chunks, schickt sie mit der Frage an das LLM und zeigt die Antwort sowie die Dateien der gefundenen Chunks.

In [ ]:
answer = await ask(agent, "What materials is the Logypal 1 pallet made from?")

## Fehlerfälle testen

Die Dokumente enthalten vier absichtlich eingebaute Fallen.

### a) Produkt ausserhalb des Sortiments

Eine gute Antwort sagt, dass die Lara-Palette in keinem Dokument vorkommt, statt eine andere Palette zu beschreiben.

In [ ]:
answer = await ask(agent, "What materials is the Lara pallet made from?")

### b) Fehlende Daten

Im Datenblatt der LogyLight-Palette sind alle LCA-Werte als "not yet available" angegeben. Die richtige Antwort sagt das, ohne eine Zahl zu erfinden.

In [ ]:
answer = await ask(agent, "What is the GWP of the LogyLight pallet?")

### c) Widersprüchliche Quellen

Das Relicyc-Datenblatt vom Juni 2021 nennt 4.1 kg CO₂e pro Palette aus einer internen, nicht verifizierten LCA. Die EPD von 2023 (S-P-09135) ist extern verifiziert. Die Antwort sollte den Widerspruch benennen und die EPD bevorzugen.

In [ ]:
answer = await ask(
    agent, "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?"
)

### d) Nicht verifizierte Lieferantenangabe

Die tesa-Broschüre nennt eine CO₂-Reduktion von 68 % aus einer internen Bewertung, ohne EPD. Die Antwort sollte die Angabe als nicht verifiziert kennzeichnen.

In [ ]:
answer = await ask(
    agent,
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?",
)

### Ein strengerer System-Prompt

`SYSTEM_PROMPT` in `baseline_rag.py` ergänzt Regeln für genau diese Fälle. Er wird ab T04 verwendet. Die Tests damit wiederholen.

Übersetzt:

> Du bist ein hilfreicher KI-Assistent, spezialisiert auf Nachhaltigkeit und Produkt-Compliance für die PrimePack AG.
>
> Du erhältst Dokumentauszüge, die für die Frage der Nutzerin oder des Nutzers relevant sind. Formuliere die bestmögliche Antwort und verwende dabei nur die Informationen aus diesen Auszügen.
>
> Regeln:
> - Verwende die bereitgestellten Auszüge als einzige Wahrheitsquelle. Stütze dich nicht auf externes Wissen.
> - Berücksichtige alle relevanten Auszüge für deine Antwort.
> - Wenn die Antwort nicht in den Auszügen steht, sage klar, dass du es nicht weisst.
> - Nenne für jede Aussage das Quelldokument mit Dateinamen.
> - Wenn Auszüge widersprüchliche Informationen enthalten, nenne beide Werte und weise auf den Widerspruch hin.
> - Unterscheide zwischen extern verifizierten Angaben (EPDs) und Selbstdeklarationen von Lieferanten.

In [ ]:
print(SYSTEM_PROMPT)

strict_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)
for query in [
    "What materials is the Lara pallet made from?",
    "What is the GWP of the LogyLight pallet?",
    "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?",
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?",
]:
    print(f"\n>>> {query}")
    answer = await ask(strict_agent, query)

## Zusammenfassung

| Schritt | Funktion |
|---|---|
| 1. Laden und chunken | `load_chunks()` |
| 2. Embedden und indexieren | `build_vector_store(chunks, embedding_model, reset=True)` |
| 3. Retrieval prüfen | `inspect_retrieval(query, vector_store, embedding_model, top_k)` |
| 4. Agent bauen | `build_agent(vector_store, embedding_model, llm, top_k, system_prompt)` |
| 5. Antworten | `ask(agent, query)` |

`run_pipeline("openai", query=...)` führt die Schritte 2 bis 5 auf dem bestehenden Store in einem Aufruf aus.